## Coverage del Tokenizer. Análisis

In [1]:

# IMPORTANTE: Configurar HF_HOME ANTES de importar datasets
import os
os.environ['HF_HOME'] = '/media/datos1/tts-valenciano/huggingface'
os.environ['HF_DATASETS_CACHE'] = '/media/datos1/tts-valenciano/huggingface/datasets'

print(f"HF_HOME establecido en: {os.environ['HF_HOME']}")
print(f"HF_DATASETS_CACHE establecido en: {os.environ['HF_DATASETS_CACHE']}")

from dotenv import load_dotenv

load_dotenv()

from huggingface_hub import login
login(os.getenv("HUGGINGFACE_TOKEN"))

HF_HOME establecido en: /media/datos1/tts-valenciano/huggingface
HF_DATASETS_CACHE establecido en: /media/datos1/tts-valenciano/huggingface/datasets


In [3]:
from transformers import BertTokenizer

# Cargar tokenizer
print("Cargando tokenizer...")
tokenizer = BertTokenizer.from_pretrained('javiimts/bert-ca-va-tokenizer')
print("Tokenizer cargado")
print(f"Vocab size: {tokenizer.vocab_size}")
print(f"Tokens especiales: {tokenizer.special_tokens_map}")

Cargando tokenizer...
Tokenizer cargado
Vocab size: 29850
Tokens especiales: {'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': 'M'}


In [5]:
import phonemizer

# Configuramos el phonemizer para probar el tokenizer
global_phonemizer = phonemizer.backend.EspeakBackend(
    language="ca-va",
    preserve_punctuation=True,
    with_stress=True,
    language_switch="remove-flags",
)

In [6]:
from datasets import load_dataset
# Cargamos dataset
dataset = load_dataset("javiimts/corts-valencianes-asr-parquet")
print(f"Dataset cargado")
print(f"Particiones disponibles: {list(dataset.keys())}")
for split in dataset.keys():
    print(f"{split}: {len(dataset[split])} muestras")

Resolving data files:   0%|          | 0/81 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/77 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/81 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/77 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/71 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/68 [00:00<?, ?it/s]

Dataset cargado
Particiones disponibles: ['clean_train_short', 'clean_test_short', 'clean_dev_short', 'other_train_short', 'other_test_short', 'other_dev_short', 'clean_train_long', 'clean_test_long', 'clean_dev_long', 'other_train_long', 'other_test_long', 'other_dev_long']
clean_train_short: 44932 muestras
clean_test_short: 1325 muestras
clean_dev_short: 1284 muestras
other_train_short: 44399 muestras
other_test_short: 1467 muestras
other_dev_short: 1267 muestras
clean_train_long: 1286 muestras
clean_test_long: 31 muestras
clean_dev_long: 31 muestras
other_train_long: 1390 muestras
other_test_long: 21 muestras
other_dev_long: 27 muestras


In [10]:
from collections import Counter, defaultdict

# Seleccionar partición para análisis
SPLIT = "clean_test_short"
NUM_SAMPLES = min(1000, len(dataset[SPLIT]))  # Analizar máximo 1000 muestras

# Métricas a recopilar
metrics = {
    'total_samples': 0,
    'total_chars_original': 0,
    'total_chars_phonemized': 0,
    'total_tokens': 0,
    'total_unks': 0,
    'samples_with_unks': 0,
    'token_lengths': [],
    'char_lengths_original': [],
    'char_lengths_phonemized': [],
    'compression_ratios': [],
    'token_counter': Counter(),
    'unk_contexts': [],  # Contextos donde aparecen UNKs
    'phoneme_counter': Counter(),
}

In [14]:
import numpy as np
from tqdm import tqdm

# Procesar muestras
print("\n   Procesando muestras...")
for i, sample in enumerate(tqdm(dataset[SPLIT].select(range(NUM_SAMPLES)), desc="   Analizando")):
    text = sample['text']
    
    # Fonemizar
    try:
        phonemized = global_phonemizer.phonemize([text], strip=True, njobs=1)[0]
    except Exception as e:
        print(f"\n   ⚠ Error al fonemizar muestra {i}: {e}")
        continue
    
    # Tokenizar
    encoded = tokenizer(phonemized, add_special_tokens=True)
    input_ids = encoded['input_ids']
    tokens = tokenizer.convert_ids_to_tokens(input_ids)
    
    # Contar métricas
    metrics['total_samples'] += 1
    metrics['total_chars_original'] += len(text)
    metrics['total_chars_phonemized'] += len(phonemized)
    metrics['total_tokens'] += len(input_ids)
    
    # Contar caracteres en el texto fonemizado
    for char in phonemized:
        if char != ' ':
            metrics['phoneme_counter'][char] += 1
    
    # Detectar UNKs
    unk_count = tokens.count(tokenizer.unk_token)
    metrics['total_unks'] += unk_count
    if unk_count > 0:
        metrics['samples_with_unks'] += 1
        # Guardar contexto
        metrics['unk_contexts'].append({
            'text': text,
            'phonemized': phonemized,
            'tokens': tokens,
            'unk_count': unk_count
        })
    
    # Guardar longitudes
    metrics['token_lengths'].append(len(input_ids))
    metrics['char_lengths_original'].append(len(text))
    metrics['char_lengths_phonemized'].append(len(phonemized))
    
    # Calcular ratio de compresión (chars/tokens)
    compression = len(phonemized) / len(input_ids) if len(input_ids) > 0 else 0
    metrics['compression_ratios'].append(compression)
    
    # Contar tokens
    for token in tokens:
        if token not in ['[CLS]', '[SEP]', '[PAD]']:
            metrics['token_counter'][token] += 1

# Calcular estadísticas
print("\n" + "="*80)
print("RESULTADOS DEL ANÁLISIS")
print("="*80)

print("\n ESTADÍSTICAS GENERALES:")
print(f"   - Muestras analizadas: {metrics['total_samples']}")
print(f"   - Total caracteres (original): {metrics['total_chars_original']:,}")
print(f"   - Total caracteres (fonemizado): {metrics['total_chars_phonemized']:,}")
print(f"   - Total tokens generados: {metrics['total_tokens']:,}")
print(f"   - Caracteres por muestra (original): {metrics['total_chars_original']/metrics['total_samples']:.1f}")
print(f"   - Caracteres por muestra (fonemizado): {metrics['total_chars_phonemized']/metrics['total_samples']:.1f}")
print(f"   - Tokens por muestra: {metrics['total_tokens']/metrics['total_samples']:.1f}")

print("\n COMPRESIÓN:")
avg_compression = np.mean(metrics['compression_ratios'])
print(f"   - Ratio promedio (chars_phonemized/tokens): {avg_compression:.2f}")
print(f"   - Esto significa que cada token representa ~{avg_compression:.2f} caracteres")
print(f"   - Ratio mínimo: {np.min(metrics['compression_ratios']):.2f}")
print(f"   - Ratio máximo: {np.max(metrics['compression_ratios']):.2f}")
print(f"   - Desviación estándar: {np.std(metrics['compression_ratios']):.2f}")

print("\n TOKENS DESCONOCIDOS (UNK):")
unk_rate = (metrics['total_unks'] / metrics['total_tokens']) * 100
samples_with_unk_rate = (metrics['samples_with_unks'] / metrics['total_samples']) * 100
print(f"   - Total de UNKs: {metrics['total_unks']:,}")
print(f"   - Tasa de UNKs: {unk_rate:.2f}%")
print(f"   - Muestras con UNKs: {metrics['samples_with_unks']} ({samples_with_unk_rate:.1f}%)")
print(f"   - Muestras sin UNKs: {metrics['total_samples'] - metrics['samples_with_unks']} ({100-samples_with_unk_rate:.1f}%)")

if metrics['samples_with_unks'] > 0:
    print(f"\n Ejemplos de contextos con UNK (primeros 5):")
    for i, ctx in enumerate(metrics['unk_contexts'][:5]):
        print(f"\n   Ejemplo {i+1}:")
        print(f"      Original: {ctx['text'][:100]}...")
        print(f"      Fonemizado: {ctx['phonemized'][:100]}...")
        print(f"      Tokens: {' '.join(ctx['tokens'][:20])}...")
        print(f"      UNKs en esta muestra: {ctx['unk_count']}")


print("JUSHUHEUHEUHE")
print("\n DISTRIBUCIÓN DE LONGITUDES:")
print(f"   Longitud de tokens:")
print(f"   - Media: {np.mean(metrics['token_lengths']):.1f}")
print(f"   - Mediana: {np.median(metrics['token_lengths']):.1f}")
print(f"   - Mínimo: {np.min(metrics['token_lengths'])}")
print(f"   - Máximo: {np.max(metrics['token_lengths'])}")
print(f"   - Percentil 95: {np.percentile(metrics['token_lengths'], 95):.0f}")
print(f"   - Percentil 99: {np.percentile(metrics['token_lengths'], 99):.0f}")

print("\n TOKENS MÁS FRECUENTES (Top 20):")
for i, (token, count) in enumerate(metrics['token_counter'].most_common(20), 1):
    percentage = (count / metrics['total_tokens']) * 100
    print(f"   {i:2d}. '{token}' - {count:,} veces ({percentage:.2f}%)")

print("\n FONEMAS MÁS FRECUENTES (Top 20):")
for i, (phoneme, count) in enumerate(metrics['phoneme_counter'].most_common(20), 1):
    percentage = (count / metrics['total_chars_phonemized']) * 100
    print(f"   {i:2d}. '{phoneme}' - {count:,} veces ({percentage:.2f}%)")


# Guardar métricas en JSON
results = {
    'general_stats': {
        'total_samples': metrics['total_samples'],
        'total_chars_original': metrics['total_chars_original'],
        'total_chars_phonemized': metrics['total_chars_phonemized'],
        'total_tokens': metrics['total_tokens'],
        'avg_chars_per_sample_original': metrics['total_chars_original']/metrics['total_samples'],
        'avg_chars_per_sample_phonemized': metrics['total_chars_phonemized']/metrics['total_samples'],
        'avg_tokens_per_sample': metrics['total_tokens']/metrics['total_samples'],
    },
    'compression': {
        'avg_ratio': float(np.mean(metrics['compression_ratios'])),
        'min_ratio': float(np.min(metrics['compression_ratios'])),
        'max_ratio': float(np.max(metrics['compression_ratios'])),
        'std_ratio': float(np.std(metrics['compression_ratios'])),
    },
    'unk_analysis': {
        'total_unks': metrics['total_unks'],
        'unk_rate_percentage': float(unk_rate),
        'samples_with_unks': metrics['samples_with_unks'],
        'samples_with_unks_percentage': float(samples_with_unk_rate),
    },
    'length_stats': {
        'tokens': {
            'mean': float(np.mean(metrics['token_lengths'])),
            'median': float(np.median(metrics['token_lengths'])),
            'min': int(np.min(metrics['token_lengths'])),
            'max': int(np.max(metrics['token_lengths'])),
            'p95': float(np.percentile(metrics['token_lengths'], 95)),
            'p99': float(np.percentile(metrics['token_lengths'], 99)),
        }
    },
    'top_tokens': dict(metrics['token_counter'].most_common(50)),
    'top_phonemes': dict(metrics['phoneme_counter'].most_common(50)),
}



# Resumen final
print("\n" + "="*80)
print("RESUMEN Y RECOMENDACIONES")
print("="*80)

print("\nCALIDAD DEL TOKENIZER:")
if unk_rate < 1:
    print(f" EXCELENTE: Tasa de UNK muy baja ({unk_rate:.2f}%)")
elif unk_rate < 5:
    print(f"BUENA: Tasa de UNK aceptable ({unk_rate:.2f}%)")
else:
    print(f"MEJORABLE: Tasa de UNK alta ({unk_rate:.2f}%)")

print(f"\n   Ratio de compresión: {avg_compression:.2f} chars/token")
if avg_compression > 3:
    print(f"Buena compresión (>3 chars/token)")
elif avg_compression > 2:
    print(f"Compresión moderada (2-3 chars/token)")
else:
    print(f"Baja compresión (<2 chars/token)")




   Procesando muestras...


   Analizando:   0%|          | 0/1000 [00:00<?, ?it/s]

   Analizando: 100%|██████████| 1000/1000 [00:01<00:00, 596.57it/s]


RESULTADOS DEL ANÁLISIS

 ESTADÍSTICAS GENERALES:
   - Muestras analizadas: 3000
   - Total caracteres (original): 348,600
   - Total caracteres (fonemizado): 376,833
   - Total tokens generados: 168,240
   - Caracteres por muestra (original): 116.2
   - Caracteres por muestra (fonemizado): 125.6
   - Tokens por muestra: 56.1

 COMPRESIÓN:
   - Ratio promedio (chars_phonemized/tokens): 2.17
   - Esto significa que cada token representa ~2.17 caracteres
   - Ratio mínimo: 1.14
   - Ratio máximo: 3.33
   - Desviación estándar: 0.26

 TOKENS DESCONOCIDOS (UNK):
   - Total de UNKs: 0
   - Tasa de UNKs: 0.00%
   - Muestras con UNKs: 0 (0.0%)
   - Muestras sin UNKs: 3000 (100.0%)
JUSHUHEUHEUHE

 DISTRIBUCIÓN DE LONGITUDES:
   Longitud de tokens:
   - Media: 56.1
   - Mediana: 46.0
   - Mínimo: 4
   - Máximo: 239
   - Percentil 95: 145
   - Percentil 99: 201

 TOKENS MÁS FRECUENTES (Top 20):
    1. '##a' - 16,536 veces (9.83%)
    2. '##ɾ' - 6,150 veces (3.66%)
    3. 'a' - 5,805 veces (3.45